### <b> Serving the Churn Model with Flask </b>

In [2]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
#!pip3 install tqdm
from tqdm.auto import tqdm
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [3]:
# Setting model parameters
C = 1.0
n_splits = 5

output_file = f'model_C={C}.bin'

In [5]:
# Data preparation

df = pd.read_csv('Customer-Churn_data.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
 
#column normalization and strings column cleaning
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)

strings_columns = list(df.dtypes[df.dtypes=='string'].index)

for col in strings_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_', regex=False)

df.churn = (df.churn.str.lower() == 'yes').astype(int)

In [6]:
# Data splitting
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=1)

numerical = ['tenure', 'monthlycharges', 'totalcharges']
 
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [7]:
# Training
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C, max_iter=3000)
    model.fit(X_train, y_train)

    return dv, model

In [8]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.fit_transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [10]:
# Validation
print(f'doing validation with C={C}')

kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []
fold = 0

for train_idx, val_idx in kfold.split(df_train_full):
    df_train = df_train_full.iloc[train_idx]
    df_val = df_train_full.iloc[val_idx]
    
    y_train = df_train.churn.values
    y_val = df_val.churn.values
        
    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)
    
    print(f'auc on fold {fold} is {auc}')
    fold += 1

print('validation result:')
print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

doing validation with C=1.0
auc on fold 0 is 0.844427785322354
auc on fold 1 is 0.844964656999073
auc on fold 2 is 0.8332570740517761
auc on fold 3 is 0.8347728944170876
auc on fold 4 is 0.851801010322864
validation result:
C=1.0 0.842 +- 0.007


In [13]:
# Train the final model
print('train the final model')

train the final model


In [15]:
dv, model = train(df_train_full, df_train_full.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)
y_test = df_test.churn.values

In [16]:
auc = roc_auc_score(y_test, y_pred)

In [17]:
print(f'auc={auc}')

auc=0.8584709176985496


In [18]:
# Saving the model with Pickle
with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)

In [19]:
print(f'the model is saved to {output_file}')

the model is saved to model_C=1.0.bin
